# Feed Forward Fully Connected Layer

In [1]:
# Load the required library
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [2]:
from datetime import datetime

In [3]:
current_time = datetime.now().strftime("%Y%m%d%H%M%S")

In [4]:
print(f"{current_time}")

20240701163734


In [5]:
progress_file = f'/group/pmc021/amunif/epi-thesis/workflow/04. Neural Network/progress_{current_time}.txt'

In [6]:
dataset_path = "/group/pmc021/amunif/epi-thesis/dataset/"

def load_large_csv(file_name, chunksize=20000):
    # Read the CSV file
    mylist = []

    for chunk in pd.read_csv(file_name, chunksize = chunksize):
        mylist.append(chunk)

    df = pd.concat(mylist, axis = 0)
    
    del mylist
    return df

In [7]:
def save_progress(file_name, message):
    with open(file_name, 'a+') as file:
        file.write(message + "\n")

In [8]:
# Load the data
X = pd.read_csv(f"{dataset_path}histone_features.csv", nrows=100)
y = pd.read_csv(f"{dataset_path}value_1_df.csv", nrows=100)

In [9]:
# Convert to numpy
X_np = X.to_numpy()
y_np = y.to_numpy()

In [10]:
# Split the data into training and testing
X_train_np, X_test_np, y_train_np, y_test_np = train_test_split(X_np, y_np, test_size=0.2, random_state=42)


In [11]:
# Convert NumPy arrays to PyTorch tensors
X_train_tensor = torch.tensor(X_train_np, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_np, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_np, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_np, dtype=torch.float32)

In [12]:
# Create TensorDataset for training and testing sets
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

In [13]:
# Create a DataLoader
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [23]:
# Neural network model

class NeuralNetwork(nn.Module):
    def __init__(self, input_dim):
        super(NeuralNetwork, self).__init__()
        self.fc1 = nn.Linear(input_dim, 1024)
        self.fc2 = nn.Linear(1024, 512)
        self.fc3 = nn.Linear(512, 256)
        self.fc4 = nn.Linear(256, 128)
        self.fc5 = nn.Linear(128, 64) 
        self.fc6 = nn.Linear(64, 1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.relu(self.fc3(x))
        x = torch.relu(self.fc4(x))
        x = torch.relu(self.fc5(x))
        x = self.fc6(x)  # No activation function for the output layer in regression
        return x

In [24]:
input_dim = 20000
model = NeuralNetwork(input_dim)
criterion = nn.MSELoss()  # Mean Squared Error loss for regression
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [25]:
# Train the model
num_epochs = 100

In [26]:
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    message = f'Epoch [{epoch + 1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}'
    print(message)
    save_progress(progress_file, message)

Epoch [1/100], Loss: 480.1976
Epoch [2/100], Loss: 499.3564
Epoch [3/100], Loss: 591.9876
Epoch [4/100], Loss: 610.3454
Epoch [5/100], Loss: 474.2509
Epoch [6/100], Loss: 598.2786
Epoch [7/100], Loss: 431.7558
Epoch [8/100], Loss: 443.0154
Epoch [9/100], Loss: 1249.8851
Epoch [10/100], Loss: 378.6700
Epoch [11/100], Loss: 1117.8649
Epoch [12/100], Loss: 317.1710
Epoch [13/100], Loss: 863.8380
Epoch [14/100], Loss: 731.6124
Epoch [15/100], Loss: 228.3181
Epoch [16/100], Loss: 206.7646
Epoch [17/100], Loss: 167.6997
Epoch [18/100], Loss: 120.4431
Epoch [19/100], Loss: 101.0876
Epoch [20/100], Loss: 92.2025
Epoch [21/100], Loss: 80.6971
Epoch [22/100], Loss: 71.0598
Epoch [23/100], Loss: 48.8869
Epoch [24/100], Loss: 30.8148
Epoch [25/100], Loss: 18.1414
Epoch [26/100], Loss: 14.1753
Epoch [27/100], Loss: 9.1592
Epoch [28/100], Loss: 7.4377
Epoch [29/100], Loss: 14.9349
Epoch [30/100], Loss: 14.6645
Epoch [31/100], Loss: 11.8290
Epoch [32/100], Loss: 9.6991
Epoch [33/100], Loss: 16.5243
E

In [27]:
# Evaluate the model
model.eval()
total_loss = 0.0
with torch.no_grad():
    for inputs, labels in train_loader:
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        total_loss += loss.item()

print(f'Average loss on the training data: {total_loss/len(train_loader):.4f}')

Average loss on the training data: 1.5376


In [28]:
# Making predictions with the test data
with torch.no_grad():
    predictions = model(X_test_tensor)

In [29]:
# Convert predictions to NumPy array
predictions_np = predictions.numpy()
y_test_np = y_test_tensor.numpy()

In [30]:
# Calculate evaluation metrics
mse = mean_squared_error(y_test_np, predictions_np)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test_np, predictions_np)
r2 = r2_score(y_test_np, predictions_np)

print(f'MSE: {mse}')
print(f'RMSE: {rmse}')
print(f'MAE: {mae}')
print(f'R2 Score: {r2}')

MSE: 5.7820940017700195
RMSE: 2.4045984745025635
MAE: 0.8871172666549683
R2 Score: 0.9968039393424988


In [31]:
# Create a DataFrame with true values and predictions
results_df = pd.DataFrame({
    'True Values': y_test_np.flatten(),
    'Predictions': predictions_np.flatten()
})

# Save the DataFrame to a CSV file
results_df.to_csv('predictions.csv', index=False)

print("Predictions and true values saved to 'predictions.csv'.")

Predictions and true values saved to 'predictions.csv'.
